In [1]:
!pip install spacy --upgrade

In [2]:
!python -m spacy download en_core_web_sm

/home/michalek/miniconda3/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.1) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 8.0 MB/s  0:00:01a 0:00:01m eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [3]:
import spacy
import en_core_web_sm
import pandas as pd
import seaborn as sns
import numpy as np
import re
import random

/home/michalek/miniconda3/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.1) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [4]:
train_data = pd.read_csv('data/train.csv', header = None,
                         names = ['sentiment', 'id', 'date', 'query', 'user', 'text'], encoding='latin1')

In [5]:
train_data

,sentiment,id,date,query,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
...,...,...,...,...,...,...
1599995,4,2193601966,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,AmandaMarie1028,Just woke up. Having no school is the best fee...
1599996,4,2193601969,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,TheWDBoards,TheWDB.com - Very cool to hear old Walt interv...
1599997,4,2193601991,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,bpbabe,Are you ready for your MoJo Makeover? Ask me f...
1599998,4,2193602064,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,tinydiamondz,Happy 38th Birthday to my boo of alll time!!! ...


In [6]:
train_data['sentiment'].unique()

array([0, 4])

In [7]:
# sns.countplot(train_data['sentiment']);

In [8]:
np.unique(train_data['sentiment'], return_counts=True)

(array([0, 4]), array([800000, 800000]))

In [9]:
train_data = train_data.drop(['id', 'date', 'query', 'user'], axis=1)
train_data

,sentiment,text
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,is upset that he can't update his Facebook by ...
2,0,@Kenichan I dived many times for the ball. Man...
3,0,my whole body feels itchy and like its on fire
4,0,"@nationwideclass no, it's not behaving at all...."
...,...,...
1599995,4,Just woke up. Having no school is the best fee...
1599996,4,TheWDB.com - Very cool to hear old Walt interv...
1599997,4,Are you ready for your MoJo Makeover? Ask me f...
1599998,4,Happy 38th Birthday to my boo of alll time!!! ...


## Train and test data

In [10]:
X = train_data.iloc[:, 1].values
X

array(["@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer.  You shoulda got David Carr of Third Day to do it. ;D",
       "is upset that he can't update his Facebook by texting it... and might cry as a result  School today also. Blah!",
       '@Kenichan I dived many times for the ball. Managed to save 50%  The rest go out of bounds',
       ..., 'Are you ready for your MoJo Makeover? Ask me for details ',
       'Happy 38th Birthday to my boo of alll time!!! Tupac Amaru Shakur ',
       'happy #charitytuesday @theNSPCC @SparksCharity @SpeakingUpH4H '],
      dtype=object)

In [12]:
y = train_data.iloc[:, 0].values
y

array([0, 0, 0, ..., 4, 4, 4])

In [13]:
from sklearn.model_selection import train_test_split
X, _, y, _ = train_test_split(X, y, test_size = 0.97)

In [14]:
X

array(['@Niki7a a very good idea ',
       'Wings r one away from the stanley cup!!  ',
       '@saultheturtle, thank you! ', ...,
       'Good morning! (to some good nite) I hope you had an enjoyable weekend ',
       '@SandyFowler Thank you for the very kind words about my latest blog post/story! Glad you enjoyed it! Sending smiles ... ',
       "I hate how Victoria Beckham spells 'colors' like 'colours' &amp; 'favors' like 'favours' "],
      dtype=object)

In [15]:
X.shape

(48000,)

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2)

In [17]:
X_train.shape, y_train.shape

((38400,), (38400,))

In [18]:
X_test.shape, y_test.shape

((9600,), (9600,))

In [19]:
np.unique(y_train, return_counts=True)

(array([0, 4]), array([19326, 19074]))

In [20]:
np.unique(y_test, return_counts=True)

(array([0, 4]), array([4759, 4841]))

# Preprocessing the data

In [21]:
nlp = spacy.load('en_core_web_sm')
nlp

In [23]:
def preprocessing(sentence):
  sentence = sentence.lower()
  sentence = re.sub(r"@[A-Za-z0-9]+", ' ', sentence)
  sentence = re.sub(r"https?://[A-Za-z0-9./]+", ' ', sentence)
  sentence = sentence.replace('.', '')
  tokens = []
  tokens = [token.text for token in nlp(sentence) if not (token.is_stop or token.like_num or token.is_punct or token.is_space or len(token) == 1)]
  tokens = ' '.join([element for element in tokens])

  return tokens

In [24]:
preprocessing("@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer.  2 You shoulda got David Carr of Third Day to do it. ;D")

'awww bummer shoulda got david carr day'

In [25]:
X_train_cleaned = [preprocessing(tweet) for tweet in X_train]

In [26]:
len(X_train_cleaned)

38400

In [27]:
for _ in range(10):
  print(X_train_cleaned[random.randint(0, len(X_train_cleaned) - 1)])

hi peter think awesome play carlisle way imagined rock
yay bj lorraine amd baby hahaha
wow day talk blame nice luv
going town today cos lazy yesterday lol gona breakfast
damn youuuuuuu want subway
yes like blipping tunes
left amp union sqi miss em
fail cheer freedoms land watch taken away
congrats bnet cnet aop ftws slightly jealous watch backs lads ladettes
cahhh transferring chino hills bit


In [28]:
X_test_cleaned = [preprocessing(tweet) for tweet in X_test]

## Word cloud

In [29]:
texts = ''
for text in X_train_cleaned:
  texts += ' ' + text

In [30]:
texts

" coffee time tried sending picture morning stayed quot;sending twitpic quot screen forever wrong nita phone stopped allowing send texts keeps failing resting long tough day beautiful wife disapointed jon amp kate trouble internet police sexual harrassment twitter fuckkkk classic course chris speak truth think disappeared answered got iphone touch hello found guys wefollow 5:30am coffee having apple instead 3turnoffwords lets friends time steve found geocache kl bit sweating sure heat walk ummmm probably known oldies lol truly love types left restraunt table air force guys uniform yay watching ass stuck road poorly francis having sleep nite amp hay fever raping real thoit looks like girl threw large shirt pinned legs called dress saying goodbye hubby called giddy hottie calls darn got shortlisted floor job guess coverletter good interview quot;get like storm keeps heading way everyday shelter&quot blue_wolf spare iphones van sorry try glad spread word got 2:2 happy happy amp amazing pe